# 12 — Chat Tutor (RAG)

This notebook demonstrates the **ChatTutor** workflow — a RAG-powered conversational
tutor that retrieves relevant knowledge chunks and generates grounded answers.

## Features
- **Retrieval-Augmented Generation**: Answers are grounded in your study materials
- **Conversation Memory**: Maintains last 5 exchanges for follow-up context
- **Source Citations**: Every answer cites the chunk IDs it draws from
- **Out-of-Scope Detection**: Gracefully handles questions outside the knowledge base
- **Follow-Up Support**: Understands pronouns and contextual references

In [ ]:
import sys
sys.path.insert(0, "..")

from src.llm import LLMClient
from src.retrieval.retriever import Retriever
from src.store import VectorStore, KnowledgeGraph
from src.workflows import ChatTutor

## Setup

Initialize the components: vector store, knowledge graph, retriever, and LLM client.

In [ ]:
# Initialize the vector store and knowledge graph
vector_store = VectorStore()
knowledge_graph = KnowledgeGraph()

# Create the retriever
retriever = Retriever(vector_store=vector_store, knowledge_graph=knowledge_graph)

# Create the LLM client
llm_client = LLMClient()

print(f"Available LLM providers: {[p.value for p in llm_client.available_providers]}")

## Create the Chat Tutor

The `ChatTutor` wraps the retriever and LLM client into a conversational interface.

In [ ]:
tutor = ChatTutor(retriever=retriever, llm_client=llm_client)
print("Chat Tutor initialized!")
print(f"History: {tutor.history}")

## Ask a Question

The `ask()` method runs the full RAG pipeline:
1. **Retrieve** relevant chunks from the knowledge base
2. **Generate** a grounded answer using the LLM
3. **Return** the answer with source citations and grounding status

In [ ]:
# Ask an initial question
result = tutor.ask("What is photosynthesis?")

print("Answer:")
print(result["answer"])
print(f"\nSources: {result['sources']}")
print(f"Grounded: {result['is_grounded']}")

## Follow-Up Questions

The tutor maintains conversation history and uses it for context-aware retrieval
and generation. Follow-up questions with pronouns or references are handled naturally.

In [ ]:
# Ask a follow-up using a pronoun reference
result = tutor.ask("How does it convert sunlight into energy?")

print("Answer:")
print(result["answer"])
print(f"\nSources: {result['sources']}")
print(f"Grounded: {result['is_grounded']}")

In [ ]:
# Another follow-up
result = tutor.ask("What role does chlorophyll play?")

print("Answer:")
print(result["answer"])
print(f"\nSources: {result['sources']}")
print(f"Grounded: {result['is_grounded']}")

## Conversation History

View the stored conversation history. The tutor keeps the last 5 exchanges.

In [ ]:
# View conversation history
print(f"History length: {len(tutor.history)} messages")
print()
for msg in tutor.history:
    role = msg['role'].upper()
    content = msg['content'][:100] + "..." if len(msg['content']) > 100 else msg['content']
    print(f"[{role}]: {content}")

## Out-of-Scope Questions

When asked about topics not covered in the knowledge base, the tutor indicates
this gracefully rather than hallucinating.

In [ ]:
# Ask something likely outside the knowledge base
result = tutor.ask("What is the current stock price of Apple?")

print("Answer:")
print(result["answer"])
print(f"\nSources: {result['sources']}")
print(f"Grounded: {result['is_grounded']}")

## Reset Conversation

Clear the conversation history to start fresh.

In [ ]:
# Reset and verify
tutor.reset()
print(f"History after reset: {tutor.history}")

# Start a new conversation
result = tutor.ask("What is machine learning?")
print(f"\nNew conversation answer:")
print(result["answer"])

## Summary

The `ChatTutor` provides:
- **RAG Pipeline**: retrieve → generate → cite sources
- **Conversation Memory**: Last 5 exchanges for follow-up support
- **Grounding**: Answers are based only on available material
- **Source Citations**: Every claim is backed by chunk references
- **Graceful Degradation**: Out-of-scope topics are flagged clearly